# 3.3 — Stacking Ensemble

This notebook builds a **diversity-aware stacking ensemble** on top of the tuned models trained in `3.2`.

It loads the results cache from `./models/3.3-all_results_cache.joblib` (produced by `3.2`) and runs:

| # | Section |
|---|---|
| 1 | Imports & Setup |
| 2 | Pipeline Helpers |
| 3 | Load Datasets & Results Cache |
| 4 | Stacking Ensemble |

## 1. Imports & Setup

In [ ]:
import subprocess, sys

_packages = [
    'polars', 'scikit-learn', 'xgboost', 'catboost', 'interpret', 'shap',
    'joblib', 'matplotlib', 'seaborn', 'jinja2',
    'numpy', 'pandas', 'optuna',
]
for pkg in _packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet', pkg])

print('All dependencies installed.')

In [ ]:
from pathlib import Path

import polars as pl
import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import (
    RandomForestRegressor,
    HistGradientBoostingRegressor,
    StackingRegressor,
)
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

try:
    from xgboost import XGBRegressor
    _xgb_available = True
except ImportError:
    _xgb_available = False

try:
    from catboost import CatBoostRegressor
    _catboost_available = True
except ImportError:
    _catboost_available = False

try:
    from interpret.glassbox import ExplainableBoostingRegressor
    _ebm_available = True
except ImportError:
    _ebm_available = False

import matplotlib.pyplot as plt
import seaborn as sns

print(f'Libraries loaded.')
print(f'  XGBoost  : {_xgb_available}')
print(f'  CatBoost : {_catboost_available}')
print(f'  EBM      : {_ebm_available}')

## 2. Pipeline Helpers

In [ ]:
_NUMERIC_DTYPES = {
    pl.Int8,  pl.Int16,  pl.Int32,  pl.Int64,
    pl.UInt8, pl.UInt16, pl.UInt32, pl.UInt64,
    pl.Float32, pl.Float64,
}
_STRING_DTYPES = {pl.Utf8, pl.String, pl.Categorical}


def build_preprocessor(numeric_cols: list, cat_cols: list) -> ColumnTransformer:
    """Build a ColumnTransformer with median imputation + scaling / one-hot."""
    numeric_transformer = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler',  StandardScaler()),
    ])
    transformers = []
    if numeric_cols:
        transformers.append(('num', numeric_transformer, numeric_cols))
    if cat_cols:
        cat_transformer = Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot',  OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ])
        transformers.append(('cat', cat_transformer, cat_cols))
    if not transformers:
        raise ValueError('No numeric or categorical columns found.')
    return ColumnTransformer(transformers=transformers, remainder='drop')


def _align_test(test_df: pl.DataFrame, feature_cols: list, X_train: pl.DataFrame) -> pl.DataFrame:
    """Align test DataFrame columns to match training feature columns."""
    aligned = pl.DataFrame()
    for col in feature_cols:
        if col in test_df.columns:
            aligned = aligned.with_columns(test_df[col])
        else:
            null_s = pl.Series([None] * test_df.shape[0], dtype=X_train[col].dtype).alias(col)
            aligned = aligned.with_columns(null_s)
    return aligned.select(feature_cols)


print('Pipeline helpers defined: build_preprocessor(), _align_test()')

In [ ]:
MODEL_CLASSES = {
    'Ridge':        Ridge,
    'Lasso':        Lasso,
    'RandomForest': RandomForestRegressor,
    'HistGBM':      HistGradientBoostingRegressor,
    'MLP':          MLPRegressor,
}
if _xgb_available:
    MODEL_CLASSES['XGBoost'] = XGBRegressor
if _catboost_available:
    MODEL_CLASSES['CatBoost'] = CatBoostRegressor
if _ebm_available:
    MODEL_CLASSES['EBM'] = ExplainableBoostingRegressor

print(f'MODEL_CLASSES: {list(MODEL_CLASSES.keys())}')

## 3. Load Datasets & Results Cache

In [ ]:
DATASETS = {
    '2.4-FeatEng': {
        'train_path': './data/interim/2.4-train.parquet',
        'test_path':  './data/interim/2.4-test.parquet',
    }
}

TARGET   = 'health_gain'
all_data = {}

for label, paths in DATASETS.items():
    train_path = Path(paths['train_path'])
    test_path  = Path(paths['test_path'])
    if not train_path.exists():
        print(f'SKIP {label} — {train_path} not found. Run the corresponding data prep notebook first.')
        continue
    train_pl  = pl.read_parquet(train_path)
    test_pl   = pl.read_parquet(test_path)
    drop_cols = ['OHS_Success'] if 'OHS_Success' in train_pl.columns else []
    has_year  = 'Year' in train_pl.columns
    all_data[label] = (train_pl, test_pl, drop_cols)
    print(f'{label:22s}  train={train_pl.shape}  test={test_pl.shape}  has_Year={has_year}  drop={drop_cols}')

print(f'\nTarget  : {TARGET}')
print(f'Loaded  : {list(all_data.keys())}')

# ── Load 3.3 results cache (produced by notebook 3.2) ────────────────────────
RESULTS_PATH_33 = Path('./models/3.3-all_results_cache.joblib')

if not RESULTS_PATH_33.exists():
    raise FileNotFoundError(
        f'Results cache not found at {RESULTS_PATH_33}.\n'
        'Run notebook 3.2 (Section 5 — Individual Tuned Runs) first to generate it.'
    )

all_results_33 = joblib.load(RESULTS_PATH_33)
print(f'\nLoaded 3.3 cache ({sum(len(v) for v in all_results_33.values())} runs)')
for ds, md in all_results_33.items():
    for m, r in md.items():
        print(f'  ✓  {ds:22s} × {m:<20s}  Test RMSE={r["test_rmse"]:.4f}  R²={r["test_r2"]:.4f}')

## 4. Stacking Ensemble — Diversity-Aware Base Learner Selection

Uses `sklearn.ensemble.StackingRegressor` with a **diversity-aware** selection of 3 base learners  
from the winning dataset, and a Ridge meta-learner.

### Why diversity over raw performance?
Picking the top-3 by RMSE often selects correlated models from the same family (e.g. three boosters).  
Stacking correlated models yields diminishing returns — the meta-learner sees nearly identical signals.

### Selection algorithm
1. **Best dataset**: identified by lowest Test RMSE across all (dataset × model) results.
2. **Performance floor**: only models within 20% RMSE of the best individual are eligible  
   (avoids selecting a diverse-but-poor model).
3. **Greedy max-diversity**: anchor on the best-RMSE model, then greedily add the candidate  
   with the lowest average Pearson correlation of **residuals** with already-selected models.
4. **Residual correlation**: measures *error alignment* — two models that fail on different samples  
   are genuinely complementary, regardless of their raw accuracy.

How stacking works:  
1. `StackingRegressor` generates out-of-fold (OOF) predictions from each base estimator using `cv=5` internally.  
2. The Ridge meta-learner is trained on those OOF predictions.  
3. Prediction = learned blend of base estimator outputs.

In [ ]:
# ── Build comparison table to identify best dataset ───────────────────────────
if not all_results_33:
    raise RuntimeError('No results in all_results_33 — run Section 5 cells in notebook 3.2 first.')

stack_rows = []
for ds_label, models_dict in all_results_33.items():
    for mdl_name, res in models_dict.items():
        stack_rows.append({'Dataset': ds_label, 'Model': mdl_name,
                            'Test RMSE': res['test_rmse'], 'Test R²': res['test_r2']})

stack_df     = pl.DataFrame(stack_rows).sort('Test RMSE')
best_dataset = stack_df.row(0, named=True)['Dataset']
print(f'Best dataset for stacking: {best_dataset}')
print(stack_df.filter(pl.col('Dataset') == best_dataset))

# ── Diversity-aware base learner selection ────────────────────────────────────
# Step 1: collect eligible models (within 20% RMSE of best individual on the best dataset)
best_ds_results  = all_results_33.get(best_dataset, {})
best_rmse        = min(r['test_rmse'] for r in best_ds_results.values())
rmse_floor       = best_rmse * 1.20   # models must be within 20% of best RMSE
PERFORMANCE_FLOOR_PCT = 20

# Exclude stacking entries from base learner candidates
candidates = {
    name: res for name, res in best_ds_results.items()
    if not name.startswith('Stacking') and res['test_rmse'] <= rmse_floor
}
print(f'\nPerformance floor: Test RMSE ≤ {rmse_floor:.4f} (best={best_rmse:.4f}, +{PERFORMANCE_FLOOR_PCT}%)')
print(f'Eligible candidates ({len(candidates)}): {list(candidates.keys())}')

# Step 2: build residual matrix (rows=samples, cols=models)
#   residual = y_pred - y_test  →  correlated residuals = similar errors on same patients
candidate_names    = list(candidates.keys())
y_test_ref         = list(candidates.values())[0]['y_test']   # same test set for all
residual_matrix    = np.column_stack([
    candidates[n]['y_pred'] - candidates[n]['y_test'] for n in candidate_names
])

# Step 3: pairwise Pearson correlation of residuals
corr_matrix = np.corrcoef(residual_matrix.T)   # shape (n_models, n_models)

# Visualise correlation heatmap
fig, ax = plt.subplots(figsize=(max(5, len(candidate_names)), max(4, len(candidate_names) - 1)))
im = ax.imshow(corr_matrix, vmin=-1, vmax=1, cmap='RdYlGn_r')
ax.set_xticks(range(len(candidate_names))); ax.set_xticklabels(candidate_names, rotation=45, ha='right')
ax.set_yticks(range(len(candidate_names))); ax.set_yticklabels(candidate_names)
plt.colorbar(im, ax=ax, label='Residual Pearson ρ')
ax.set_title(f'Residual Correlation — {best_dataset}\n(lower ρ = more diverse = better for stacking)')
for i in range(len(candidate_names)):
    for j in range(len(candidate_names)):
        ax.text(j, i, f'{corr_matrix[i, j]:.2f}', ha='center', va='center', fontsize=8,
                color='white' if abs(corr_matrix[i, j]) > 0.6 else 'black')
plt.tight_layout()
plt.show()

# Step 4: greedy max-diversity selection
#   Anchor = best-RMSE model, then add the candidate with lowest avg correlation to selected set
idx_by_rmse = sorted(range(len(candidate_names)),
                     key=lambda i: candidates[candidate_names[i]]['test_rmse'])

selected_idx = [idx_by_rmse[0]]   # anchor: best RMSE model
remaining    = list(set(range(len(candidate_names))) - {selected_idx[0]})

while len(selected_idx) < min(3, len(candidate_names)):
    # For each remaining candidate, compute average correlation with already-selected models
    avg_corr = {}
    for r_idx in remaining:
        avg_corr[r_idx] = float(np.mean([corr_matrix[r_idx, s_idx] for s_idx in selected_idx]))
    # Pick the one with the LOWEST average correlation → most diverse
    next_idx = min(avg_corr, key=avg_corr.get)
    selected_idx.append(next_idx)
    remaining.remove(next_idx)
    print(f'  + Added: {candidate_names[next_idx]:<20s}  avg_corr_with_selected={avg_corr[next_idx]:.3f}')

diverse_top3 = [(candidate_names[i], candidates[candidate_names[i]]) for i in selected_idx]

# Step 5: report model families for sanity check
MODEL_FAMILIES = {
    'Ridge': 'Linear', 'Lasso': 'Linear',
    'RandomForest': 'Bagging',
    'HistGBM': 'Boosting', 'XGBoost': 'Boosting', 'CatBoost': 'Boosting', 'EBM': 'Boosting',
    'MLP': 'Neural Network',
}

print(f'\n>>> Diversity-selected base learners for stacking:')
families_selected = []
for name, res in diverse_top3:
    family = MODEL_FAMILIES.get(name, 'Unknown')
    families_selected.append(family)
    print(f'  {name:<20s}  Family={family:<15s}  Test RMSE={res["test_rmse"]:.4f}  R²={res["test_r2"]:.4f}')
    print(f'    Best params: {res["best_params"]}')

unique_families = set(families_selected)
if len(unique_families) == 1:
    print(f'\n  ⚠  WARNING: all 3 selected models are from the same family ({unique_families.pop()}).')
    print(     '     Consider relaxing the performance floor or training more model types.')
else:
    print(f'\n  ✓ Model family diversity: {", ".join(sorted(unique_families))}')

In [ ]:
# Rebuild fresh (unfitted) pipelines with tuned params for StackingRegressor
# StackingRegressor clones the estimators internally; building fresh avoids
# any fitted-state pollution.
base_estimators = []
for mdl_name, res in diverse_top3:
    fresh_model = MODEL_CLASSES[mdl_name](**res['best_params'])
    fresh_pre   = build_preprocessor(res['numeric_cols'], res['cat_cols'])
    fresh_pipe  = Pipeline([('pre', fresh_pre), ('model', fresh_model)])
    base_estimators.append((mdl_name, fresh_pipe))
    print(f'  Base estimator prepared: {mdl_name}')

stacking = StackingRegressor(
    estimators=base_estimators,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
    passthrough=False,   # meta-learner sees only base-model predictions
)

# Prepare raw training / test DataFrames for the best dataset
train_pl_best, test_pl_best, drop_best = all_data[best_dataset]
all_drop_best     = [c for c in ([TARGET] + drop_best) if c in train_pl_best.columns]
feature_cols_best = [c for c in train_pl_best.columns if c not in all_drop_best]

X_train_stack = train_pl_best.select(feature_cols_best).to_pandas()
y_train_stack = train_pl_best[TARGET].to_numpy()
X_test_stack  = _align_test(test_pl_best, feature_cols_best,
                             train_pl_best.select(feature_cols_best)).to_pandas()
y_test_stack  = test_pl_best[TARGET].to_numpy()

print(f'\nFitting StackingRegressor on {best_dataset} (may take a few minutes)...')
stacking.fit(X_train_stack, y_train_stack)

y_pred_stack    = stacking.predict(X_test_stack)
test_rmse_stack = float(np.sqrt(mean_squared_error(y_test_stack, y_pred_stack)))
test_r2_stack   = float(r2_score(y_test_stack, y_pred_stack))
test_mae_stack  = float(mean_absolute_error(y_test_stack, y_pred_stack))

best_individual_rmse = diverse_top3[0][1]['test_rmse']
improvement          = best_individual_rmse - test_rmse_stack

print(f'\n>>> Diversity-Stacking Ensemble ({best_dataset})')
print(f'    Base learners : {[n for n, _ in diverse_top3]}')
print(f'    Test MAE      : {test_mae_stack:.4f}')
print(f'    Test RMSE     : {test_rmse_stack:.4f}  (best individual: {best_individual_rmse:.4f})')
print(f'    Test R²       : {test_r2_stack:.4f}')
print(f'    Stacking vs best individual: {improvement:+.4f} RMSE  ({100*improvement/best_individual_rmse:+.1f}%)')

# Save stacking result (key includes selection method for traceability)
all_results_33.setdefault(best_dataset, {})['Stacking-Diverse'] = {
    'model_name': 'Stacking-Diverse',
    'best_params': {'base_models': [n for n, _ in diverse_top3], 'selection': 'greedy_min_residual_corr'},
    'pipeline': stacking,
    'X_train': train_pl_best.select(feature_cols_best),
    'X_test':  _align_test(test_pl_best, feature_cols_best, train_pl_best.select(feature_cols_best)),
    'y_train': y_train_stack, 'y_test': y_test_stack, 'y_pred': y_pred_stack,
    'feature_names': feature_cols_best, 'numeric_cols': feature_cols_best, 'cat_cols': [],
    'cv_mae': float('nan'), 'cv_rmse': float('nan'), 'cv_r2': float('nan'),
    'test_mae': test_mae_stack, 'test_rmse': test_rmse_stack, 'test_r2': test_r2_stack,
}
joblib.dump(all_results_33, RESULTS_PATH_33)
print(f'\n✓ Diversity-stacking result saved → {RESULTS_PATH_33}')